**Lane:** Lane 2 — Refresh / Content Opportunity Scoring. **Lane confirmed, locking this week.**

This week I do on my lane what the session did live on FlyRank's product flags: check that the
signals a rule leans on are real (with a bucket table and n, and an honest verdict — a
clearly-explained negative counts as a win), encode ONE transparent rule with ONE reason code and
an action label, write the ranked queue, and read my own top ten with a skeptic's eye. This is
the baseline my Week-5 model has to beat.

**Data note:** this notebook runs on the starter dataset shipped in this repo
(`data/raw/content_refresh_anonymized.csv`, 30,000 rows / 32 clients), the same one `w01`/`w02`
used. `w03`'s data contract moved onto the gated warehouse (`hf://datasets/FlyRank/internship-warehouse`)
for later, heavier querying work — this week's job is the transparent baseline itself, and the
starter CSV already carries the exact fields the session's own rule-building script
(`scripts/02_baseline_score.py`) uses: `days_since_last_update`, `avg_position`, `ctr`,
`impressions_last_30d`. No warehouse query is needed to build or verify a rule a human can read.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RamyaMuthumariappan/FlyRankAI.git"
REPO_DIR = "FlyRankAI"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import numpy as np
import pandas as pd
pd.set_option("display.width", 140)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# is_declining_label is carried along for POST-HOC reading only (section 3/4 commentary,
# and a future Week-5 evaluation) — it is never an input to anything scored below.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Rows, columns:", df.shape)
print("Base decline rate (whole catalogue, for later comparison):", round(df["is_declining_label"].mean(), 3))

Working dir: /content/FlyRankAI
Rows, columns: (30000, 45)
Base decline rate (whole catalogue, for later comparison): 0.542


## 1. Two signal checks, then my rule

The session's live build showed three FlyRank product flags and the plain signal each one leans
on: staleness behind the refresh flags, CTR-vs-position behind the CTR-fix logic, and volume
behind quick-win. My rule idea leans on the first two, so those are the two I check here —
**before** I let either one drive a score. One bucket table each, n printed, one-word verdict:
CONFIRMED / OPPOSITE / MIXED / FALSE.

### Signal check 1 — staleness behind the `stale_visible_page` refresh flag

**Claim:** a page that has gone longer without an update is more likely to be a declining page —
the logic behind FlyRank's `stale_visible_page` flag (`days_since_last_update >= 180` AND
`impressions_90d >= 500` in the session's own script).

**Test:** bucket `freshness_tier` (the data's own staleness buckets) against the decline-rate
proxy (`is_declining_label`), with n per bucket. Then check the literal product-flag threshold
on its own, since a bucket table can hide a rare-but-real flag.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
order = ["0-30", "31-90", "91-180", "181+"]
df["freshness_tier"] = pd.Categorical(df["freshness_tier"], categories=order, ordered=True)

signal1_table = df.groupby("freshness_tier", observed=True).agg(
    n=("content_id", "size"),
    decline_rate=("is_declining_label", "mean"),
    mean_impressions_last_30d=("impressions_last_30d", "mean"),
).round(3)
print("Staleness bucket vs decline rate (n printed, floor is ~50 rows):")
print(signal1_table.to_string())
print()
print(f"Overall decline rate for comparison: {df['is_declining_label'].mean():.3f}  (n={len(df):,})")

# Now the literal product-flag threshold, on its own:
flag = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
n_flag = int(flag.sum())
print()
print(f"Literal stale_visible_page flag (days>=180 & impressions_90d>=500): n={n_flag}")
if n_flag < 50:
    print(f"  -> n={n_flag} is below the ~50-row verdict floor. Too small to trust on its own.")

verdict_1 = "MIXED"
print()
print(f"VERDICT: {verdict_1}")
print("Why: the two biggest, safely-above-floor buckets (0-30d, n=20,480 and 91-180d, n=9,171)")
print("do show a real 10-point gap (0.511 -> 0.611) — staleness is doing SOMETHING. But the trend")
print("does not extend: the most-stale bucket (181+, n=174) drops back to 0.471, and its mean")
print("impressions (~108) are far below every other bucket (~1,200-2,100) — pages that go truly")
print("stale mostly stop getting traffic too, so there is little left to 'decline'. And the")
print("literal product-flag threshold only matches 17 rows here, too few to verdict at all.")
print("Reading: staleness alone is not a reliable rule input. It only means something when a")
print("page is STILL visible — so my rule uses staleness gated on visibility, never staleness alone.")


Staleness bucket vs decline rate (n printed, floor is ~50 rows):
                    n  decline_rate  mean_impressions_last_30d
freshness_tier                                                
0-30            20480         0.511                   1178.084
31-90             175         0.589                   2120.086
91-180           9171         0.611                   2001.391
181+              174         0.471                    108.184

Overall decline rate for comparison: 0.542  (n=30,000)

Literal stale_visible_page flag (days>=180 & impressions_90d>=500): n=17
  -> n=17 is below the ~50-row verdict floor. Too small to trust on its own.

VERDICT: MIXED
Why: the two biggest, safely-above-floor buckets (0-30d, n=20,480 and 91-180d, n=9,171)
do show a real 10-point gap (0.511 -> 0.611) — staleness is doing SOMETHING. But the trend
does not extend: the most-stale bucket (181+, n=174) drops back to 0.471, and its mean
impressions (~108) are far below every other bucket (~1,200-2,100) —

### Signal check 2 — CTR-vs-position behind the `low_ctr_visible_page` CTR-fix flag

**Claim:** click-through rate drops as average position gets worse — the pattern the CTR-fix
logic assumes (a page in a good spot with unusually low CTR is a fixable snippet/title problem,
not just a ranking problem).

**Test:** bucket `position_tier` (the data's own tiers, ordered top_3 -> deep) against CTR,
computed the honest way — weighted (`sum(clicks)/sum(impressions)`), not the mean of a per-row
rate, per the auditing-signals rule about averaging rates. n per bucket, rows with
`avg_position == 0` ("no data", not rank zero) excluded first.


In [3]:
valid_pos = df[df["avg_position"] > 0].copy()  # avg_position == 0 means "no data", not rank zero
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
valid_pos["position_tier"] = pd.Categorical(valid_pos["position_tier"], categories=pos_order, ordered=True)

grp = valid_pos.groupby("position_tier", observed=True)
signal2_table = pd.DataFrame({
    "n": grp.size(),
    "weighted_ctr_pct": (grp["clicks_90d"].sum() / grp["impressions_90d"].sum() * 100).round(3),
})
print("Position tier vs weighted CTR (n printed):")
print(signal2_table.to_string())

verdict_2 = "CONFIRMED"
print()
print(f"VERDICT: {verdict_2}")
print("Why: weighted CTR falls monotonically from top_3 (0.489%) through page_1 (0.350%),")
print("striking (0.347%), page_3_5 (0.155%), down to deep (0.041%) — every bucket is safely")
print("above the ~50-row floor (smallest is n=1,116). The CTR-fix flag's underlying assumption")
print("holds cleanly in this data, so a page that sits at a GOOD position but well below its")
print("tier's own CTR benchmark is a real, checkable opportunity — this is the stronger of my")
print("two signals, and my rule weights it accordingly.")

Position tier vs weighted CTR (n printed):
                   n  weighted_ctr_pct
position_tier                         
top_3           1116             0.489
page_1         11814             0.350
striking        7304             0.347
page_3_5        7242             0.155
deep            1319             0.041

VERDICT: CONFIRMED
Why: weighted CTR falls monotonically from top_3 (0.489%) through page_1 (0.350%),
striking (0.347%), page_3_5 (0.155%), down to deep (0.041%) — every bucket is safely
above the ~50-row floor (smallest is n=1,116). The CTR-fix flag's underlying assumption
holds cleanly in this data, so a page that sits at a GOOD position but well below its
tier's own CTR benchmark is a real, checkable opportunity — this is the stronger of my
two signals, and my rule weights it accordingly.


### My rule and its reason codes

**In plain words:** a page is worth reviewing first if it's earning meaningful search visibility
but converting that visibility poorly — either its click-through rate sits well below what a page
at its position should earn (the CONFIRMED signal, weighted higher), or it's aging and stale
while still drawing real traffic (the MIXED signal, so it only counts when paired with
visibility, never alone).

**Score** (readable on purpose, no fitted weights — hand-picked constants that mirror how much
I trust each signal from the checks above):

```
visible       = impressions_last_30d >= 50
stale_flag    = days_since_last_update >= 90       # the natural split between this data's two big tiers
ctr_gap_flag  = 0 < avg_position <= 20  AND  ctr < 0.35   # below the striking-tier CTR benchmark found above

score = 0.6 * ctr_gap_flag * visible * log1p(impressions_last_30d)
      + 0.4 * stale_flag   * visible * log1p(impressions_last_30d)
```

**ONE reason code per row** (priority order — CTR gap wins ties, since it's the confirmed
signal):

| Priority | Condition | `reason_code` | `action_label` |
|---|---|---|---|
| 1 | `ctr_gap_flag` & `visible` | `low_ctr_visible_page` | `refresh_and_review_ctr` |
| 2 | `stale_flag` & `visible` | `stale_visible_page` | `refresh` |
| else | — | `general_monitor` | `monitor` |

No future-window or label-derived input anywhere in this: `trend_direction`, `trend_pct`,
`is_declining_label` never appear on the right-hand side of the score.


## 2. Build the ranked queue (writes the CSV)

Code the score and reason code exactly as specified above, rank the full catalogue, and write
`work/outputs/baseline_action_score.csv` — the file the CI leak-guard expects to be regenerated,
not committed.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = (df["impressions_last_30d"] >= 50).astype(int)
stale_flag = (df["days_since_last_update"] >= 90).astype(int)
ctr_gap_flag = ((df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.35)).astype(int)

df["baseline_action_score"] = (
    0.6 * ctr_gap_flag * visible * np.log1p(df["impressions_last_30d"])
    + 0.4 * stale_flag * visible * np.log1p(df["impressions_last_30d"])
)

df["reason_code"] = np.select(
    [(ctr_gap_flag == 1) & (visible == 1), (stale_flag == 1) & (visible == 1)],
    ["low_ctr_visible_page", "stale_visible_page"],
    default="general_monitor",
)
df["action_label"] = np.select(
    [df["reason_code"] == "low_ctr_visible_page", df["reason_code"] == "stale_visible_page"],
    ["refresh_and_review_ctr", "refresh"],
    default="monitor",
)

# LEAKAGE GUARD: assert none of the label-derived columns were used above.
score_inputs = {"impressions_last_30d", "days_since_last_update", "avg_position", "ctr"}
banned = {"trend_direction", "trend_pct", "is_declining_label", "priority_tier"}
assert score_inputs.isdisjoint(banned), "score input list overlaps a banned column — stop."
print("Leakage guard: score built only from", sorted(score_inputs), "- no banned column touched.")

ranked = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

print()
print("reason_code counts:")
print(ranked["reason_code"].value_counts().to_string())
print()
print("action_label counts:")
print(ranked["action_label"].value_counts().to_string())
print()
print("score describe():")
print(ranked["baseline_action_score"].describe().round(3).to_string())

out_cols = [
    "rank", "content_id", "client_id", "baseline_action_score", "reason_code", "action_label",
    "avg_position", "ctr", "impressions_last_30d", "days_since_last_update",
    "content_type", "is_declining_label",  # carried for future eval only, not used above
]
out_path = "work/outputs/baseline_action_score.csv"
os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv(out_path, index=False)
print()
print(f"Wrote ranked queue: {out_path}  ({len(ranked):,} rows)")


Leakage guard: score built only from ['avg_position', 'ctr', 'days_since_last_update', 'impressions_last_30d'] - no banned column touched.

reason_code counts:
reason_code
general_monitor         17292
low_ctr_visible_page     8780
stale_visible_page       3928

action_label counts:
action_label
monitor                   17292
refresh_and_review_ctr     8780
refresh                    3928

score describe():
count    30000.000
mean         1.741
std          2.325
min          0.000
25%          0.000
50%          0.000
75%          3.337
max         11.714

Wrote ranked queue: work/outputs/baseline_action_score.csv  (30,000 rows)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)

def position_phrase(pos):
    if pos <= 3:
        return "top 3"
    if pos <= 10:
        return "page 1"
    if pos <= 20:
        return "striking distance (page 2)"
    return "beyond page 2"

for _, row in top20.iterrows():
    why = (
        f"position {row['avg_position']:.1f} ({position_phrase(row['avg_position'])}) but CTR is only "
        f"{row['ctr']:.2f}% against a ~0.35%+ benchmark for that spot, on "
        f"{row['impressions_last_30d']:,} impressions/30d — real, wasted visibility."
    )
    trend_note = (
        "the trend proxy also reads 'declining' here, so a CTR fix and a broader content refresh "
        "may both be worth doing"
        if row["is_declining_label"] == 1
        else "the trend proxy currently reads 'not declining' — so if the CTR gap turns out to be "
             "seasonal or snippet-driven rather than a real title/meta problem, a CTR-only fix "
             "could be chasing nothing"
    )
    would_be_wrong = (
        f"this pick would be wrong if the low CTR is structural rather than fixable — e.g. a "
        f"featured snippet or AI overview above it is absorbing clicks regardless of title/meta "
        f"quality, or the query intent genuinely doesn't want a click (a definitional answer). "
        f"Also worth a caveat: {trend_note}."
    )
    print(f"Rank {int(row['rank']):>2} | {row['content_id']} | action: {row['action_label']}")
    print(f"  Why it's there: {why}")
    print(f"  What would make it wrong: {would_be_wrong}")
    print()


Rank  1 | content_4a6607efcb46 | action: refresh_and_review_ctr
  Why it's there: position 2.2 (top 3) but CTR is only 0.01% against a ~0.35%+ benchmark for that spot, on 122,303 impressions/30d — real, wasted visibility.
  What would make it wrong: this pick would be wrong if the low CTR is structural rather than fixable — e.g. a featured snippet or AI overview above it is absorbing clicks regardless of title/meta quality, or the query intent genuinely doesn't want a click (a definitional answer). Also worth a caveat: the trend proxy currently reads 'not declining' — so if the CTR gap turns out to be seasonal or snippet-driven rather than a real title/meta problem, a CTR-only fix could be chasing nothing.

Rank  2 | content_5fe46e04994d | action: refresh_and_review_ctr
  Why it's there: position 4.2 (page 1) but CTR is only 0.14% against a ~0.35%+ benchmark for that spot, on 120,791 impressions/30d — real, wasted visibility.
  What would make it wrong: this pick would be wrong if the 

## 4. Weak picks + leakage check

Which picks look wrong, and why — plus a final confirmation that no product flag or
future-window column ever touched the score.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Weak pick 1: a borderline-position pick inside the top 10 ---
weak = top20[top20["avg_position"] > 15]
print("Weak pick check — any top-10 rows with position > 15 (near the position<=20 cutoff)?")
print(weak[["rank", "content_id", "avg_position", "ctr", "impressions_last_30d"]].to_string(index=False))
print("These are the shakiest of the ten: avg_position is a 30-day AVERAGE, so a page sitting")
print("near the edge of the <=20 cutoff could be swinging in and out of 'good enough position'")
print("day to day — the flag can fire on an average that no single day actually looked like.")

print()
print("-" * 70)
print()

# --- Weak pick 2: the rule's own structural blind spot ---
first_stale_rank = int(ranked.loc[ranked["reason_code"] == "stale_visible_page", "rank"].min())
top30_reasons = ranked.head(30)["reason_code"].value_counts()
print("Structural weak point in the RULE itself, not just one row:")
print(f"  first stale_visible_page pick doesn't appear until rank {first_stale_rank:,}")
print(f"  top-30 reason codes: {dict(top30_reasons)}")
print("  The 0.6/0.4 weighting was meant to let both signals compete, but both terms share the")
print("  same log1p(impressions) multiplier — so at the huge-impression end of the catalogue,")
print("  a 0.6x CTR-gap term always outscores a 0.4x staleness term with similar volume. My")
print("  entire top 30 is one reason code. A stronger version (matching how the session's own")
print("  score used percentile_rank(), not raw log1p, before combining terms) would normalize")
print("  each component to [0,1] first so the two signals can actually trade off — worth fixing")
print("  before Week 5, not silently carrying into the model comparison.")

print()
print("-" * 70)
print()

# --- Final leakage confirmation ---
banned_cols = {"trend_direction", "trend_pct", "is_declining_label", "priority_tier"}
score_formula_vars = {"ctr_gap_flag", "stale_flag", "visible", "impressions_last_30d",
                       "avg_position", "ctr", "days_since_last_update"}
print("Leakage check: score formula only ever referenced:", sorted(score_formula_vars))
print("None of the banned label-derived columns", banned_cols, "were inputs.")
print("is_declining_label is present in the output CSV as a column, but only for a FUTURE")
print("precision@K evaluation against this frozen baseline — never as a scoring input here.")

Weak pick check — any top-10 rows with position > 15 (near the position<=20 cutoff)?
 rank           content_id  avg_position  ctr  impressions_last_30d
    7 content_eb366e871254          16.6  0.2                 66003
These are the shakiest of the ten: avg_position is a 30-day AVERAGE, so a page sitting
near the edge of the <=20 cutoff could be swinging in and out of 'good enough position'
day to day — the flag can fire on an average that no single day actually looked like.

----------------------------------------------------------------------

Structural weak point in the RULE itself, not just one row:
  first stale_visible_page pick doesn't appear until rank 3,685
  top-30 reason codes: {'low_ctr_visible_page': np.int64(30)}
  The 0.6/0.4 weighting was meant to let both signals compete, but both terms share the
  same log1p(impressions) multiplier — so at the huge-impression end of the catalogue,
  a 0.6x CTR-gap term always outscores a 0.4x staleness term with similar volume. My

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.